# Qwen 2.5 Coder 7B → GGUF → Q4_0 Quantization

This notebook demonstrates:

1. Downloading Qwen 2.5 Coder 7B Instruct from Hugging Face.
2. Converting the model into GGUF format.
3. Quantizing the model into Q4_0 format.
4. Preparing the model for Android deployment.

## 1. Install HF CLI

In [ ]:
!pip install -U "huggingface_hub[cli]"

## 2.  Download Model

In [ ]:
from pathlib import Path
import subprocess

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
OUTPUT_DIR = "./qwen-2.5-7b"

cmd = [
    "hf",
    "download",
    MODEL_NAME,
    "--local-dir",
    OUTPUT_DIR,
    "--exclude",
    "*.bin",
    "--max-workers",
    "8"
]

subprocess.run(cmd)

## 3. Verify Download

In [ ]:
from pathlib import Path

model_dir = Path("./qwen-2.5-7b")

if model_dir.exists():
    for file in model_dir.iterdir():
        print(file.name)
else:
    print("Model directory not found")

## 4. Clone llama.cpp

In [ ]:
from pathlib import Path
import subprocess

if not Path("llama.cpp").exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ggml-org/llama.cpp.git"]
    )
else:
    print("llama.cpp already exists")

## 5. Install Dependencies

In [ ]:
!pip install -r llama.cpp/requirements.txt

## 6. Convert HF → GGUF

In [ ]:
import subprocess

cmd = [
    "python",
    "llama.cpp/convert_hf_to_gguf.py",
    "./qwen-2.5-7b",
    "--outfile",
    "qwen2.5-coder-7b-f16.gguf",
    "--outtype",
    "f16"
]

subprocess.run(cmd)

## 7. Verify GGUF

In [ ]:
from pathlib import Path

gguf_file = Path("qwen2.5-coder-7b-f16.gguf")

print(f"Exists: {gguf_file.exists()}")
print(f"Size (GB): {gguf_file.stat().st_size / 1024**3:.2f}")

Expected: ~14 GB

## 8. Download llama.cpp Quantization Tools

This notebook uses the prebuilt quantization binaries from llama.cpp release `b9761`.

Release:
https://github.com/ggml-org/llama.cpp/releases/tag/b9761

Steps:

1. Download the appropriate archive for your platform.
2. Extract the archive.
3. Copy `llama-quantize.exe` into the notebook working directory (Windows) or add it to PATH.
4. Verify the binary is accessible before proceeding.

Expected binary:

- Windows: `llama-quantize.exe`
- Linux/macOS: `llama-quantize`

In [ ]:
from pathlib import Path
import platform

quantizer = (
    Path("llama.cpp/llama-quantize.exe")
    if platform.system() == "Windows"
    else Path("llama.cpp/llama-quantize")
)

print(f"Checking: {quantizer}")

if quantizer.exists():
    print("✓ Quantizer found")
else:
    print("✗ Quantizer not found")

## 9. Quantization

Windows:

In [ ]:
import subprocess

cmd = [
    "./llama.cpp/llama-quantize.exe",
    "qwen2.5-coder-7b-f16.gguf",
    "qwen2.5-coder-7b-q4_0.gguf",
    "Q4_0"
]

subprocess.run(cmd)

Linux:

In [ ]:
import subprocess

cmd = [
    "./llama.cpp/llama-quantize",
    "qwen2.5-coder-7b-f16.gguf",
    "qwen2.5-coder-7b-q4_0.gguf",
    "Q4_0"
]

subprocess.run(cmd)

## 10. Verify Quantized Model

In [ ]:
from pathlib import Path

gguf = Path("qwen2.5-coder-7b-q4_0.gguf")

print(f"Exists: {gguf.exists()}")
print(f"Size (GB): {gguf.stat().st_size / 1024**3:.2f}")

Expected: 4.2–4.5 GB

## 11. Summary

## Results

| Stage | Output |
|---------|---------|
| HF Download | Success |
| GGUF Conversion | Success |
| Q4_0 Quantization | Success |

Final Artifact: qwen2.5-coder-7b-q4_0.gguf

Approximate Size: 4.3 GB
